# IDM Bulk Data Pull — concurrent extraction + analysis

Pull millions of records out of PingIDM / ForgeRock IDM using concurrent threads, then analyze the data in pandas.

**Workflow:** input CSV of ids → `ThreadPoolExecutor` GET calls (rich progress bar) → output CSV → pandas DataFrame → analysis.

> All hostnames, credentials, and ids here are fake. Copy `notebooks/idm_config.example.json` to your own config and never commit real credentials.

## 1. Imports
`requests` for the HTTP session (with urllib3 retry adapter), `urllib.parse` for safe URL building, `concurrent.futures` for the thread pool, `rich` for the progress bar, `pandas` for analysis.

In [ ]:
import csv, json
import urllib.parse
from concurrent.futures import ThreadPoolExecutor

import pandas as pd
import requests
from rich.progress import Progress, SpinnerColumn, TextColumn, BarColumn, TaskProgressColumn, TimeElapsedColumn

from idm_pull.client import IdmClient
from idm_pull.extract import read_input_ids, extract
from idm_pull import analysis

pd.set_option('display.max_columns', 50)
print('libraries loaded')

## 2. Configuration
Point at your IDM, pick the resource, the input id column, and the fields you want in the output CSV.

In [ ]:
CONFIG = {
    "base_url": "https://idm.example.com:8443/openidm",  # <-- your IDM
    "username": "svc-data-extract",                      # <-- service account
    "password": "REPLACE-ME",                            # <-- use env vars / vault in real runs
    "resource": "managed/user",
    "input_csv": "notebooks/input_ids.example.csv",
    "id_column": "userName",
    "fields": ["userName", "mail", "givenName", "sn", "employeeNumber",
               "accountStatus", "city", "country"],
    "output_csv": "idm_extract.csv",
    "max_workers": 10,
    "rate_limit_per_second": 20.0,
}
FIELDS = CONFIG['fields']
print(f"pulling {len(FIELDS)} fields from {CONFIG['resource']}")

## 3. Connect
One shared `requests.Session` with connection pooling + automatic retry on 429/502/503/504, and a cross-thread rate limiter so the pool never trips the server throttle.

In [ ]:
client = IdmClient(
    base_url=CONFIG['base_url'],
    username=CONFIG['username'],
    password=CONFIG['password'],
    rate_limit_per_second=CONFIG['rate_limit_per_second'],
    max_retries=3, backoff_factor=1.0, page_size=100,
)
# sanity check: read one object through urllib.parse-built URL
probe = client.read(CONFIG['resource'], 'example.jdoe', fields=['userName', 'mail'])
print('connected OK, probe:', probe.get('userName'))

## 4. Read the input ids
One identifier per row in the input CSV (e.g. `userName`). This is how you drive a pull of millions of records — feed the ids, the thread pool does the rest.

In [ ]:
ids = read_input_ids(CONFIG['input_csv'], CONFIG['id_column'])
print(f'{len(ids)} ids to pull')
ids[:5]

## 5. Pull — ThreadPoolExecutor + rich progress
`extract()` fans `client.read` across `max_workers` threads, shows a live `rich` progress bar, writes the output CSV row-by-row (so a crash doesn't lose everything), and hands you back a DataFrame. Per-record failures are captured in the `_error` column, never fatal.

In [ ]:
df = extract(
    client=client,
    resource=CONFIG['resource'],
    input_ids=ids,
    fields=FIELDS,
    output_csv=CONFIG['output_csv'],
    max_workers=CONFIG['max_workers'],
)
print(df.shape)
df.head()

## 6. Analysis — the data is already in a DataFrame
First-look checks: field completeness, duplicate/fake-profile leads, top values, fetch errors.

In [ ]:
# how complete is each field?
analysis.completeness(df, FIELDS)

In [ ]:
# duplicate identity leads — same mail/employeeNumber appearing more than once
summary, dup_rows = analysis.duplicates(df, 'mail')
print(f"{len(summary)} duplicated mail values")
summary.head(10)

In [ ]:
summary_emp, _ = analysis.duplicates(df, 'employeeNumber')
print(f"{len(summary_emp)} duplicated employeeNumber values")
summary_emp.head(10)

In [ ]:
# top values — e.g. which city/country dominate the pull
analysis.value_top(df, 'country')

In [ ]:
# fetch failures — bad input ids, deleted objects, permission issues
analysis.errors(df)

## 7. Your turn
From here the DataFrame is yours: joins against HR feeds, group-membership pivots, stale-account detection (`accountStatus` vs last login), or feed the dup leads into the [forgerock-data-toolkit](https://github.com/vamsh1x/forgerock-data-toolkit) remediator.

**Safety notes:** always dry-run remediation, respect the rate limiter, and never commit `idm_config.json` with real credentials — the `.gitignore` already excludes it.